# Daily Brief Builder

Front page: weather timeline + today's tasks + today's workout + countdowns + comics + music/soundtrack + news headlines.

Everything on one page — each news group scrolls as its own section. Light/dark toggle at the top (remembers your choice).

Config at the top. All build logic below. GitHub Actions runs this daily.


## 1 · Configuration

Everything you'd change over time lives here.


In [15]:
CONFIG = {
    "owner_name": "Mayank",
    "masthead_title": "THE DAILY BRIEF",
    "timezone": "Asia/Kolkata",

    # Hide task titles in Notion by checking a "Private" checkbox column.
    # If the column doesn't exist on the database, this does nothing (so safe to leave on).
    "mask_private_tasks": True,

    # ---- Workout ----
    # Must match the Notion "Loadout" multi-select option exactly.
    "loadout": "1 · Full Gym",

    # ---- Weather ----
    # Open-Meteo, no key needed. Mumbai coords by default.
    "weather": {
        "name": "Mumbai",
        "latitude": 19.0760,
        "longitude": 72.8777,
        "units": "metric",  # "metric" or "imperial"
    },

    # ---- Countdowns ----
    # Each shows "N days left" until the given date.
    "countdowns": [
        {"name": "90-Day Fitness Challenge", "date": "2026-12-31"},
    ],

    # ---- Webcomic RSS feeds ----
    # Per-feed lookback and item cap.
    "webcomic_feeds": [
        {"name": "xkcd",  "url": "https://xkcd.com/rss.xml",            "lookback_days": 10,  "items": 2},
        {"name": "oglaf",   "url": "https://www.oglaf.com/feeds/rss/",      "lookback_days": 2, "items": 2},
        {"name": "IRO",   "url": "https://www.irovedout.com/feed/",      "lookback_days": 21, "items": 2},
    ],

    # ---- Music ----
    # MusicBrainz for release detection, Spotify artist ID for the "soundtrack of the day"
    # fallback embed. Find the Spotify ID from the artist's open.spotify.com URL
    # (https://open.spotify.com/artist/XXXXX -> "XXXXX"). Leave spotify_id blank to skip.
    "music_artists": [
        {"name": "Of Monsters and Men", "spotify_id": "4dwdTW1Lfiq0cM8nBAqIIz"},
        {"name": "Woodkid",              "spotify_id": "44TGR1CzjKBxSHsSEy7bi9"},
        {"name": "Sia",                  "spotify_id": "5WUlDfRSoLAfcVSX1WnrxN"},
    ],
    "music_lookback_days": 60,

    # ---- News ----
    # Each feed renders as its own section on the front page: a lead headline + card grid.
    # Multiple queries get merged (deduped by URL) into one section, which is how you
    # consolidate AI-adjacent chatter into one block.
    "news_feeds": [
        {"name": "AI",            "queries": ["Artificial Intelligence", "Claude Anthropic", "ChatGPT"], "items": 5},
        {"name": "Witcher",       "queries": ["Witcher"],                                                "items": 3},
        {"name": "GTA 6",         "queries": ["GTA 6"],                                                  "items": 3},
        {"name": "Harry Potter",  "queries": ["Harry Potter"],                                           "items": 3},
    ],
    "news_lookback_days": 2,
    "news_hl": "en-IN",
    "news_gl": "IN",
    "news_ceid": "IN:en",
}


## 2 · Imports


In [16]:
import os
import re
import json
import time
import hashlib
from pathlib import Path
from datetime import datetime, timedelta, timezone, date
from zoneinfo import ZoneInfo
from urllib.parse import quote_plus

import requests
import feedparser
from dateutil import parser as dateparser
from jinja2 import Template


## 3 · Paths and secrets

Tokens come from env vars in CI, from `.txt` files locally.


In [17]:
HERE = Path.cwd()
PROJECT_ROOT = HERE if (HERE / "scripts").exists() else HERE.parent

SCRIPTS_DIR = PROJECT_ROOT / "scripts"
OUTPUT_DIR = PROJECT_ROOT / "docs"
OUTPUT_DIR.mkdir(exist_ok=True)

def load_token(env_name: str, filename: str) -> str:
    if env_name in os.environ and os.environ[env_name].strip():
        return os.environ[env_name].strip()
    return (PROJECT_ROOT / filename).read_text(encoding="utf-8").strip()

NOTION_TASKS_TOKEN = load_token("NOTION_TASKS_TOKEN", "notion_tasks_token.txt")
NOTION_WORKOUT_TOKEN = load_token("NOTION_WORKOUT_TOKEN", "notion_workoutdash_token.txt")

TASKS_DATA_SOURCE_ID = "2e2bec68-aefa-81be-bfda-000b7366ef2d"
WORKOUT_DATA_SOURCE_ID = "28676ddd-d907-4643-aa88-10ffd7e38889"

NOTION_VERSION = "2026-03-11"


## 4 · Date helpers


In [18]:
TZ = ZoneInfo(CONFIG["timezone"])
NOW = datetime.now(TZ)
TODAY = NOW.date()
WEEKDAY_NAME = NOW.strftime("%A")

def ordinal(n: int) -> str:
    if 11 <= n % 100 <= 13:
        return f"{n}th"
    return f"{n}{ {1:'st', 2:'nd', 3:'rd'}.get(n % 10, 'th') }"

DATE_LONG = f"{NOW.strftime('%A')}, {NOW.strftime('%B')} {ordinal(NOW.day)}, {NOW.year}"
DATE_SHORT = NOW.strftime("%Y-%m-%d")
VOLUME = NOW.year - 2025
ISSUE = NOW.timetuple().tm_yday

def slugify(s: str) -> str:
    out = re.sub(r"[^a-z0-9]+", "-", s.lower()).strip("-")
    return out or "section"

def fmt_date(dt, with_year: bool = True, with_time: bool = False) -> str:
    """Cross-platform 'Oct 3, 2026 09:30' style formatting.
    strftime %-d (no leading zero) is Unix-only; Windows uses %#d. Avoid both."""
    s = f"{dt.strftime('%b')} {dt.day}"
    if with_year:
        s += f", {dt.year}"
    if with_time:
        s += f", {dt.strftime('%H:%M')}"
    return s

print(f"Building brief for {DATE_LONG} ({WEEKDAY_NAME}), loadout {CONFIG['loadout']!r}")


Building brief for Sunday, October 4th, 2026 (Sunday), loadout '1 · Full Gym'


## 5 · Notion client


In [19]:
NOTION_API = "https://api.notion.com/v1"

def notion_query(data_source_id: str, token: str, filter_body: dict | None = None) -> list[dict]:
    headers = {
        "Authorization": f"Bearer {token}",
        "Notion-Version": NOTION_VERSION,
        "Content-Type": "application/json",
    }
    url = f"{NOTION_API}/data_sources/{data_source_id}/query"
    results = []
    payload: dict = {}
    if filter_body:
        payload["filter"] = filter_body
    while True:
        r = requests.post(url, headers=headers, json=payload, timeout=30)
        r.raise_for_status()
        data = r.json()
        results.extend(data["results"])
        if not data.get("has_more"):
            break
        payload["start_cursor"] = data["next_cursor"]
    return results

def plain_text(rich: list[dict]) -> str:
    return "".join(part.get("plain_text", "") for part in (rich or []))


## 6 · Today's tasks

If `mask_private_tasks` is on and a `Private` checkbox column exists on the Tasks database, those rows render as `(private task)` instead of their title. Everything else works the same.


In [20]:
def fetch_tasks() -> list[dict]:
    rows = notion_query(
        TASKS_DATA_SOURCE_ID,
        NOTION_TASKS_TOKEN,
        filter_body={"property": "Task Date", "date": {"equals": DATE_SHORT}},
    )
    tasks = []
    for row in rows:
        props = row["properties"]
        title = plain_text(props["Task"]["title"]) or "(untitled)"
        done = props["Done"]["checkbox"]
        type_val = (props["Type"]["select"] or {}).get("name")

        is_private = False
        if CONFIG["mask_private_tasks"] and "Private" in props:
            is_private = props["Private"].get("checkbox", False)
        if is_private:
            title = "(private task)"

        tasks.append({
            "title": title,
            "done": done,
            "type": type_val,
            "type_class": re.sub(r"[^A-Za-z]", "", type_val or ""),
        })
    priority_order = {"PRIORITY": 0, "WORK": 1, "PERSONAL": 2, "RANGE": 3, "RECURRENT": 4, None: 5}
    tasks.sort(key=lambda t: (t["done"], priority_order.get(t["type"], 9), t["title"].lower()))
    return tasks

tasks = fetch_tasks()
print(f"Tasks: {len(tasks)}")


Tasks: 4


## 7 · Today's workout


In [21]:
def fetch_workouts() -> list[dict]:
    rows = notion_query(
        WORKOUT_DATA_SOURCE_ID,
        NOTION_WORKOUT_TOKEN,
        filter_body={"and": [
            {"property": "Day", "multi_select": {"contains": WEEKDAY_NAME}},
            {"property": "Loadout", "multi_select": {"contains": CONFIG["loadout"]}},
            {"property": "In Rotation", "checkbox": {"equals": True}},
        ]},
    )
    workouts = []
    for row in rows:
        p = row["properties"]
        workouts.append({
            "name": plain_text(p["Name"]["title"]) or "(untitled)",
            "type": (p["Type"]["select"] or {}).get("name"),
            "sets": p["Sets"]["number"],
            "reps": plain_text(p["Reps"]["rich_text"]),
            "weight_kg": p["Current Weight (kg)"]["number"],
            "weight_lb": (p["Current Weight (lb)"]["formula"] or {}).get("number"),
            "rest": p["Rest (sec)"]["number"],
            "muscles": [opt["name"] for opt in p["Muscle Groups"]["multi_select"]],
        })
    type_order = {"Strength": 0, "Core": 1, "Cardio": 2, "Mobility": 3, None: 4}
    workouts.sort(key=lambda w: type_order.get(w["type"], 9))
    return workouts

workouts = fetch_workouts()
print(f"Workouts: {len(workouts)}")


Workouts: 0


## 8 · Weather

Open-Meteo, no key. Pulls today's hourly temps + precip, sunrise/sunset, min/max.


In [22]:
WMO_LABELS = {
    0: ("Clear", "☀️"),
    1: ("Mostly clear", "🌤️"),
    2: ("Partly cloudy", "⛅"),
    3: ("Overcast", "☁️"),
    45: ("Fog", "🌫️"), 48: ("Fog", "🌫️"),
    51: ("Drizzle", "🌧️"), 53: ("Drizzle", "🌧️"), 55: ("Drizzle", "🌧️"),
    61: ("Rain", "🌧️"), 63: ("Rain", "🌧️"), 65: ("Heavy rain", "⛈️"),
    71: ("Snow", "🌨️"), 73: ("Snow", "🌨️"), 75: ("Heavy snow", "❄️"),
    80: ("Showers", "🌦️"), 81: ("Showers", "🌦️"), 82: ("Heavy showers", "⛈️"),
    95: ("Thunderstorm", "⛈️"), 96: ("Thunderstorm", "⛈️"), 99: ("Thunderstorm", "⛈️"),
}

def wmo_label(code: int) -> tuple[str, str]:
    return WMO_LABELS.get(code, ("", "🌡️"))

def fetch_weather() -> dict:
    w = CONFIG["weather"]
    temp_unit = "fahrenheit" if w["units"] == "imperial" else "celsius"
    unit_label = "°F" if w["units"] == "imperial" else "°C"
    params = {
        "latitude": w["latitude"],
        "longitude": w["longitude"],
        "hourly": "temperature_2m,precipitation_probability,weathercode",
        "daily": "sunrise,sunset,temperature_2m_max,temperature_2m_min,weathercode",
        "timezone": CONFIG["timezone"],
        "forecast_days": 1,
        "temperature_unit": temp_unit,
    }
    r = requests.get("https://api.open-meteo.com/v1/forecast", params=params, timeout=30)
    r.raise_for_status()
    data = r.json()

    hourly_times = data["hourly"]["time"]
    hourly_temps = data["hourly"]["temperature_2m"]
    hourly_precip = data["hourly"]["precipitation_probability"]
    hourly_codes = data["hourly"]["weathercode"]

    # Keep every 3rd hour -> 8 cells for a 24h day.
    cells = []
    for i, t in enumerate(hourly_times):
        if i % 3 != 0:
            continue
        dt = datetime.fromisoformat(t).replace(tzinfo=TZ)
        label, emoji = wmo_label(hourly_codes[i])
        cells.append({
            "hour": dt.strftime("%H:00"),
            "temp": round(hourly_temps[i]),
            "precip": hourly_precip[i],
            "emoji": emoji,
            "label": label,
            "position_pct": (dt.hour / 24) * 100,
        })

    daily = data["daily"]
    sunrise_dt = datetime.fromisoformat(daily["sunrise"][0]).replace(tzinfo=TZ)
    sunset_dt = datetime.fromisoformat(daily["sunset"][0]).replace(tzinfo=TZ)
    day_code = daily["weathercode"][0]
    day_label, day_emoji = wmo_label(day_code)

    return {
        "location": w["name"],
        "unit": unit_label,
        "min": round(daily["temperature_2m_min"][0]),
        "max": round(daily["temperature_2m_max"][0]),
        "summary": day_label,
        "emoji": day_emoji,
        "cells": cells,
        "sunrise": sunrise_dt.strftime("%H:%M"),
        "sunset": sunset_dt.strftime("%H:%M"),
        "sunrise_pct": (sunrise_dt.hour + sunrise_dt.minute / 60) / 24 * 100,
        "sunset_pct":  (sunset_dt.hour  + sunset_dt.minute  / 60) / 24 * 100,
    }

weather = fetch_weather()
print(f"Weather: {weather['summary']} {weather['min']}-{weather['max']}{weather['unit']} in {weather['location']}")


Weather: Mostly clear 26-36°C in Mumbai


## 9 · Countdowns


In [23]:
def fetch_countdowns() -> list[dict]:
    out = []
    for item in CONFIG["countdowns"]:
        target = date.fromisoformat(item["date"])
        delta = (target - TODAY).days
        if delta < 0:
            label = f"{abs(delta)} days ago"
        elif delta == 0:
            label = "Today!"
        elif delta == 1:
            label = "Tomorrow"
        else:
            label = f"{delta} days"
        out.append({
            "name": item["name"],
            "label": label,
            "date_pretty": fmt_date(target),
            "days_left": delta,
        })
    return out

countdowns = fetch_countdowns()
for c in countdowns:
    print(f"  {c['name']}: {c['label']} ({c['date_pretty']})")


  90-Day Fitness Challenge: 88 days (Dec 31, 2026)


## 10 · Webcomics (per-feed lookback)


In [24]:
def entry_dt(entry):
    for attr in ("published", "updated", "created"):
        v = entry.get(attr)
        if v:
            return dateparser.parse(v).astimezone(TZ)
    return None

def fetch_comics() -> list[dict]:
    out = []
    for feed_cfg in CONFIG["webcomic_feeds"]:
        cutoff = NOW - timedelta(days=feed_cfg["lookback_days"])
        parsed = feedparser.parse(feed_cfg["url"])
        kept = 0
        for entry in parsed.entries:
            dt = entry_dt(entry)
            if dt is None or dt < cutoff:
                continue
            out.append({
                "source": feed_cfg["name"],
                "title": entry.get("title", "Untitled strip"),
                "url": entry.get("link", "#"),
                "when": fmt_date(dt, with_year=False),
                "_dt": dt,
            })
            kept += 1
            if kept >= feed_cfg["items"]:
                break
    out.sort(key=lambda c: c["_dt"], reverse=True)
    for c in out:
        c.pop("_dt", None)
    return out

comics = fetch_comics()
print(f"Comics: {len(comics)}")


Comics: 2


## 11 · Music releases via MusicBrainz


In [25]:
MB_API = "https://musicbrainz.org/ws/2"
MB_HEADERS = {"User-Agent": "DailyBrief/1.0 (personal use)"}

def mb_get(path: str, params: dict) -> dict:
    r = requests.get(f"{MB_API}/{path}", params={**params, "fmt": "json"}, headers=MB_HEADERS, timeout=30)
    r.raise_for_status()
    return r.json()

def find_artist_mbid(name: str) -> str | None:
    data = mb_get("artist", {"query": f'artist:"{name}"', "limit": 1})
    artists = data.get("artists", [])
    return artists[0]["id"] if artists else None

def fetch_music_releases() -> list[dict]:
    cutoff = (NOW - timedelta(days=CONFIG["music_lookback_days"])).date()
    out = []
    for artist in CONFIG["music_artists"]:
        name = artist["name"]
        mbid = find_artist_mbid(name)
        time.sleep(1.1)  # MusicBrainz asks for 1 req/sec.
        if not mbid:
            continue
        data = mb_get("release-group", {"artist": mbid, "limit": 25, "offset": 0})
        time.sleep(1.1)
        for rg in data.get("release-groups", []):
            date_str = rg.get("first-release-date") or ""
            if not date_str:
                continue
            try:
                rel_date = dateparser.parse(date_str).date()
            except (ValueError, TypeError):
                continue
            if rel_date < cutoff:
                continue
            out.append({
                "artist": name,
                "title": rg.get("title", "Untitled"),
                "type": rg.get("primary-type") or "Release",
                "date": fmt_date(rel_date),
                "url": f"https://musicbrainz.org/release-group/{rg['id']}",
                "_dt": rel_date,
            })
    out.sort(key=lambda m: m["_dt"], reverse=True)
    for m in out:
        m.pop("_dt", None)
    return out

music_releases = fetch_music_releases()
print(f"Music releases: {len(music_releases)}")


Music releases: 0


## 12 · Spotify soundtrack of the day

Picks one artist from `music_artists` deterministically by day-of-year so you get a different one each morning. Rendered as a Spotify embed. Only shown when there are no new releases.

Browser autoplay-with-sound is blocked unless the user has already interacted with the page in this session — so the embed shows up ready, you click play.


In [26]:
def pick_soundtrack() -> dict | None:
    with_ids = [a for a in CONFIG["music_artists"] if a.get("spotify_id")]
    if not with_ids:
        return None
    chosen = with_ids[ISSUE % len(with_ids)]
    return {
        "artist": chosen["name"],
        "spotify_id": chosen["spotify_id"],
        "embed_url": f"https://open.spotify.com/embed/artist/{chosen['spotify_id']}?utm_source=generator&theme=0",
    }

soundtrack = pick_soundtrack() if not music_releases else None
if soundtrack:
    print(f"Soundtrack: {soundtrack['artist']}")
elif music_releases:
    print("Skipping soundtrack (new releases to feature)")
else:
    print("No Spotify IDs configured; soundtrack section will be empty")


Soundtrack: Woodkid


## 13 · News feeds (grouped)

Each entry in `news_feeds` fires one query per `queries` item, merges the results, dedupes by URL, caps to `items` on the front page and `news_items_on_topic_page` on the topic page.


In [27]:
def google_news_url(topic: str) -> str:
    return (
        f"https://news.google.com/rss/search?q={quote_plus(topic)}"
        f"&hl={CONFIG['news_hl']}&gl={CONFIG['news_gl']}&ceid={CONFIG['news_ceid']}"
    )

def parse_news_item(entry) -> dict | None:
    dt = entry_dt(entry)
    if dt is None:
        return None
    title = entry.get("title", "Untitled")
    source = ""
    if " - " in title:
        title, source = title.rsplit(" - ", 1)
    return {
        "title": title,
        "url": entry.get("link", "#"),
        "source": source or "Google News",
        "when": fmt_date(dt, with_year=False, with_time=True),
        "_dt": dt,
    }

def fetch_news() -> list[dict]:
    cutoff = NOW - timedelta(days=CONFIG["news_lookback_days"])
    feeds = []
    for feed_cfg in CONFIG["news_feeds"]:
        seen_urls = set()
        merged = []
        for query in feed_cfg["queries"]:
            parsed = feedparser.parse(google_news_url(query))
            for entry in parsed.entries:
                item = parse_news_item(entry)
                if item is None or item["_dt"] < cutoff:
                    continue
                if item["url"] in seen_urls:
                    continue
                seen_urls.add(item["url"])
                merged.append(item)
        merged.sort(key=lambda n: n["_dt"], reverse=True)
        for n in merged:
            n.pop("_dt", None)
        feeds.append({
            "name": feed_cfg["name"],
            "slug": slugify(feed_cfg["name"]),
            "entries": merged[:feed_cfg["items"]],
            "total_found": len(merged),
        })
    return feeds

news = fetch_news()
for f in news:
    print(f"  [{f['name']}] {len(f['entries'])} shown ({f['total_found']} found)")


  [AI] 5 shown (158 found)
  [Witcher] 3 shown (20 found)
  [GTA 6] 3 shown (55 found)
  [Harry Potter] 3 shown (17 found)


## 14 · Render front page + topic pages


In [28]:
template_main = Template((SCRIPTS_DIR / "template_html.html").read_text(encoding="utf-8"))
template_css = (SCRIPTS_DIR / "template_css.css").read_text(encoding="utf-8")

main_html = template_main.render(
    masthead_title=CONFIG["masthead_title"],
    owner_name=CONFIG["owner_name"],
    date_long=DATE_LONG,
    date_short=DATE_SHORT,
    volume=VOLUME,
    issue=ISSUE,
    weekday=WEEKDAY_NAME,
    loadout=CONFIG["loadout"],
    built_at=NOW.strftime("%Y-%m-%d %H:%M %Z"),
    css=template_css,
    weather=weather,
    tasks=tasks,
    workouts=workouts,
    countdowns=countdowns,
    comics=comics,
    music_releases=music_releases,
    soundtrack=soundtrack,
    news=news,
)
(OUTPUT_DIR / "daily_brief.html").write_text(main_html, encoding="utf-8")
print(f"Wrote docs/daily_brief.html ({len(main_html):,} bytes)")

# index.html: redirect so the bare Pages URL opens today's brief.
(OUTPUT_DIR / "index.html").write_text(
    '<!doctype html><meta charset="utf-8">'
    '<meta http-equiv="refresh" content="0; url=./daily_brief.html">'
    '<title>Redirecting...</title>'
    '<a href="./daily_brief.html">Today\'s brief</a>',
    encoding="utf-8",
)
print("Wrote docs/index.html")

print("\nDone.")


Wrote docs/daily_brief.html (31,841 bytes)
Wrote docs/index.html

Done.
